In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!pip install nvcc4jupyter

In [3]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpn449itbe".


In [4]:
%%cuda

#include <stdio.h>

__global__ void hello_from_gpu() {
    int blockId = blockIdx.x;
    int threadId = threadIdx.x;
    int globalId = blockId * blockDim.x + threadId;
    printf("Hello from GPU! Block ID: %d, Thread ID: %d, Global ID: %d\n", blockId, threadId, globalId);
}

int main(){
    //4 blocks, 8 threads per block
    hello_from_gpu<<<4, 8>>>();
    cudaDeviceSynchronize();
    return 0;
}

Hello from GPU! Block ID: 1, Thread ID: 0, Global ID: 8
Hello from GPU! Block ID: 1, Thread ID: 1, Global ID: 9
Hello from GPU! Block ID: 1, Thread ID: 2, Global ID: 10
Hello from GPU! Block ID: 1, Thread ID: 3, Global ID: 11
Hello from GPU! Block ID: 1, Thread ID: 4, Global ID: 12
Hello from GPU! Block ID: 1, Thread ID: 5, Global ID: 13
Hello from GPU! Block ID: 1, Thread ID: 6, Global ID: 14
Hello from GPU! Block ID: 1, Thread ID: 7, Global ID: 15
Hello from GPU! Block ID: 0, Thread ID: 0, Global ID: 0
Hello from GPU! Block ID: 0, Thread ID: 1, Global ID: 1
Hello from GPU! Block ID: 0, Thread ID: 2, Global ID: 2
Hello from GPU! Block ID: 0, Thread ID: 3, Global ID: 3
Hello from GPU! Block ID: 0, Thread ID: 4, Global ID: 4
Hello from GPU! Block ID: 0, Thread ID: 5, Global ID: 5
Hello from GPU! Block ID: 0, Thread ID: 6, Global ID: 6
Hello from GPU! Block ID: 0, Thread ID: 7, Global ID: 7
Hello from GPU! Block ID: 2, Thread ID: 0, Global ID: 16
Hello from GPU! Block ID: 2, Thread ID: 1

In [6]:
%%cuda
#include <stdio.h>        // Include standard C library for printf

// CUDA kernel function that runs on the GPU
// __global__ means this function is executed on the GPU but called from the CPU
__global__ void addOne(int *data)
{
    int i = threadIdx.x;  // Get the index of the current thread inside the block

    // Print which thread is working on which array element
    printf("GPU Block %d Thread %d processing element %d\n", blockIdx.x, threadIdx.x, i);

    // Read value from global memory, add 1, and write it back to global memory
    data[i] = data[i] + 1;
}

// Main function runs on the CPU (host)
int main()
{
    int h_data[8] = {1,2,3,4,5,6,7,8};   // Create an array in CPU memory (host memory)

    int *d_data;                         // Pointer that will point to GPU memory (device memory)

    // Allocate memory on the GPU global memory for 8 integers
    cudaMalloc(&d_data, 8*sizeof(int));

    // Copy data from CPU memory (h_data) to GPU memory (d_data)
    cudaMemcpy(d_data, h_data, 8*sizeof(int), cudaMemcpyHostToDevice);

    // Launch the CUDA kernel
    // specific instruction in CUDA
    // <<<1,8>>> means:
    // 1 block
    // 8 threads in that block
    addOne<<<1,8>>>(d_data);

    // Wait until the GPU finishes executing the kernel
    cudaDeviceSynchronize();

    // Copy the updated data from GPU memory back to CPU memory
    cudaMemcpy(h_data, d_data, 8*sizeof(int), cudaMemcpyDeviceToHost);

    // Print a message before showing results
    printf("\nFinal result:\n");

    // Loop through the array and print each element
    for(int i=0;i<8;i++)
        printf("%d ", h_data[i]);

    printf("\n");   // Print a newline

    // Free the memory allocated on the GPU
    cudaFree(d_data);
}

GPU Block 0 Thread 0 processing element 0
GPU Block 0 Thread 1 processing element 1
GPU Block 0 Thread 2 processing element 2
GPU Block 0 Thread 3 processing element 3
GPU Block 0 Thread 4 processing element 4
GPU Block 0 Thread 5 processing element 5
GPU Block 0 Thread 6 processing element 6
GPU Block 0 Thread 7 processing element 7

Final result:
2 3 4 5 6 7 8 9 



# shared memory

In [9]:
%%cuda

#include <stdio.h>         // Include standard C library for printf

// CUDA kernel that demonstrates shared memory usage
__global__ void addOneShared(int *data)
{
    // Declare shared memory array
    // Shared memory is fast memory shared by threads within the same block
    __shared__ int temp[8];

    int i = threadIdx.x;   // Get the thread index inside the block

    // Print which thread is processing which element
    printf("GPU Block %d Thread %d processing element %d\n", blockIdx.x, threadIdx.x, i);

    // Copy data from global memory (data) into shared memory (temp)
    temp[i] = data[i];

    // Synchronize all threads in the block
    // Ensures every thread has copied its data before computation begins
    __syncthreads();

    // Perform computation in shared memory
    // Each thread adds 1 to its value
    temp[i] = temp[i] + 1;

    // Synchronize again before copying data back to global memory
    __syncthreads();

    // Copy the updated value from shared memory back to global memory
    data[i] = temp[i];
}

// Main function running on CPU
int main()
{
    int h_data[8] = {1,2,3,4,5,6,7,8};   // Input array stored in CPU memory

    int *d_data;                         // Pointer for GPU memory

    // Allocate memory on GPU global memory
    cudaMalloc(&d_data, 8*sizeof(int));

    // Copy input data from CPU memory to GPU memory
    cudaMemcpy(d_data, h_data, 8*sizeof(int), cudaMemcpyHostToDevice);

    // Launch kernel with 1 block and 8 threads
    addOneShared<<<1,8>>>(d_data);

    // Wait for GPU to complete execution
    cudaDeviceSynchronize();

    // Copy results from GPU memory back to CPU memory
    cudaMemcpy(h_data, d_data, 8*sizeof(int), cudaMemcpyDeviceToHost);

    // Print heading
    printf("\nFinal result:\n");

    // Print resulting array values
    for(int i=0;i<8;i++)
        printf("%d ", h_data[i]);

    printf("\n");  // Print newline

    // Free GPU memory
    cudaFree(d_data);
}

GPU Block 0 Thread 0 processing element 0
GPU Block 0 Thread 1 processing element 1
GPU Block 0 Thread 2 processing element 2
GPU Block 0 Thread 3 processing element 3
GPU Block 0 Thread 4 processing element 4
GPU Block 0 Thread 5 processing element 5
GPU Block 0 Thread 6 processing element 6
GPU Block 0 Thread 7 processing element 7

Final result:
2 3 4 5 6 7 8 9 

